# ETL Challenge — Proof of Concept Walkthrough

This notebook demonstrates the full pipeline end-to-end:
1. Raw mock data
2. Pydantic entrance gate (Layer 1 QA)
3. PySpark join + aggregation (transforms)
4. Four post-load audit checks (Layer 2 QA)
5. JSON audit report
6. Mocked alert dispatch

In [1]:
import os
import sys

# Set JAVA_HOME so PySpark can find the JVM inside this Jupyter kernel.
# Jupyter does not inherit shell rc files, so we set it explicitly here.
os.environ.setdefault(
    "JAVA_HOME",
    os.path.expanduser("~/java/jdk-17.0.11+9/Contents/Home"),
)
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ.get("PATH", "")
print("JAVA_HOME:", os.environ["JAVA_HOME"])

JAVA_HOME: /Users/jcospinom/java/jdk-17.0.11+9/Contents/Home


## 1. Raw Mock Data

The mock data is intentionally dirty.  Every QA check is designed to surface at least one failure.

In [2]:
from etl_challenge.data.mock_data import CUSTOMERS_RAW, TRANSACTIONS_RAW

print('=== CUSTOMERS_RAW ===')
for row in CUSTOMERS_RAW:
    print(row)

print('\n=== TRANSACTIONS_RAW ===')
for row in TRANSACTIONS_RAW:
    print(row)

=== CUSTOMERS_RAW ===
{'customer_id': 1, 'name': 'Ana Torres', 'email': 'ana@email.com', 'country': 'Colombia'}
{'customer_id': 2, 'name': 'Juan Pérez', 'email': None, 'country': 'Mexico'}
{'customer_id': 3, 'name': 'Laura Gómez', 'email': 'laura_gomez@email.com', 'country': None}
{'customer_id': 4, 'name': 'Juan Pérez', 'email': 'juanperez@email.com', 'country': 'Mexico'}
{'customer_id': 5, 'name': None, 'email': 'andres@email.com', 'country': 'Chile'}

=== TRANSACTIONS_RAW ===
{'transaction_id': 100, 'customer_id': 1, 'amount': 200.0, 'date': '2025-01-01'}
{'transaction_id': 101, 'customer_id': 2, 'amount': 150.0, 'date': '2025-01-02'}
{'transaction_id': 102, 'customer_id': 2, 'amount': 150.0, 'date': '2025-01-02'}
{'transaction_id': 103, 'customer_id': 3, 'amount': None, 'date': '2025-01-03'}
{'transaction_id': 104, 'customer_id': 6, 'amount': 300.0, 'date': '2025-01-04'}


## 2. Pydantic Entrance Gate (Layer 1 QA)

Records that fail the contract are rejected immediately — they never reach Spark.
Each rejected record carries a `rejection_reason` key explaining why it failed.

In [3]:
from etl_challenge.ingestion.loader import load_and_validate

ingestion = load_and_validate(CUSTOMERS_RAW, TRANSACTIONS_RAW)

print(f'Clean customers  : {len(ingestion.clean_customers)}')
print(f'Rejected customers: {len(ingestion.rejected_customers)}')
print(f'Clean transactions  : {len(ingestion.clean_transactions)}')
print(f'Rejected transactions: {len(ingestion.rejected_transactions)}')

print('\n--- Rejected customers ---')
for r in ingestion.rejected_customers:
    print(r)

print('\n--- Rejected transactions ---')
for r in ingestion.rejected_transactions:
    print(r)

Clean customers  : 2
Rejected customers: 3
Clean transactions  : 4
Rejected transactions: 1

--- Rejected customers ---
{'customer_id': 2, 'name': 'Juan Pérez', 'email': None, 'country': 'Mexico', 'rejection_reason': '1 validation error for Customer\nemail\n  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]\n    For further information visit https://errors.pydantic.dev/2.12/v/string_type'}
{'customer_id': 3, 'name': 'Laura Gómez', 'email': 'laura_gomez@email.com', 'country': None, 'rejection_reason': '1 validation error for Customer\ncountry\n  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]\n    For further information visit https://errors.pydantic.dev/2.12/v/string_type'}
{'customer_id': 5, 'name': None, 'email': 'andres@email.com', 'country': 'Chile', 'rejection_reason': '1 validation error for Customer\nname\n  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]\n  

## 3. PySpark Transforms

Clean records are joined (transactions ← customers) and aggregated by country.

In [4]:
from etl_challenge.transforms.pipeline import run_pipeline

joined_df, aggregated_df = run_pipeline(
    ingestion.clean_customers,
    ingestion.clean_transactions,
)

print('=== Joined DataFrame ===')
joined_df.show(truncate=False)

print('=== Aggregated spend per country ===')
aggregated_df.show(truncate=False)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/26 22:34:29 WARN Utils: Your hostname, Juans-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.10.32.170 instead (on interface en0)
26/03/26 22:34:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 22:34:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


=== Joined DataFrame ===


+-----------+------+----------+--------------+--------+-------------+----------+
|customer_id|amount|date      |transaction_id|country |email        |name      |
+-----------+------+----------+--------------+--------+-------------+----------+
|1          |200.0 |2025-01-01|100           |Colombia|ana@email.com|Ana Torres|
+-----------+------+----------+--------------+--------+-------------+----------+

=== Aggregated spend per country ===


+--------+------------+
|country |total_amount|
+--------+------------+
|Colombia|200.0       |
+--------+------------+



## 4. Post-Load Audit Checks (Layer 2 QA)

Four checks run on the Spark DataFrames.  At least one will fail because the mock
data was designed to contain referential and reconciliation issues even after the
gate has removed the most obviously broken records.

In [5]:
from pyspark.sql import SparkSession

from etl_challenge.audit.checks import (
    check_completeness,
    check_reconciliation,
    check_referential_integrity,
    check_uniqueness,
)

spark = SparkSession.builder.getOrCreate()
customers_df = spark.createDataFrame(ingestion.clean_customers)
transactions_df = spark.createDataFrame(ingestion.clean_transactions)

audit_results = [
    check_completeness(customers_df, transactions_df),
    check_uniqueness(customers_df),
    check_referential_integrity(transactions_df, customers_df),
    check_reconciliation(CUSTOMERS_RAW, TRANSACTIONS_RAW, aggregated_df),
]

for result in audit_results:
    status = 'PASS' if result.passed else 'FAIL'
    print(f'[{status}] {result.check_name}: {result.details}')


[PASS] completeness: {'null_counts': {'name': 0, 'email': 0, 'country': 0, 'amount': 0}}
[PASS] uniqueness: {'duplicate_emails': []}
[FAIL] referential_integrity: {'orphan_customer_ids': [2, 6]}
[FAIL] reconciliation: {'raw_count': 5, 'transformed_count': 1, 'delta': 4}


## 5. JSON Audit Report

All results are assembled into a structured JSON report and written to disk.

In [6]:
import json

from etl_challenge.reporting.report import build_report, save_report

rejected_counts = {
    'customers': len(ingestion.rejected_customers),
    'transactions': len(ingestion.rejected_transactions),
}

report = build_report(audit_results, rejected_counts)
save_report(report, 'audit_report.json')

print(json.dumps(report, indent=2))

{
  "timestamp": "2026-03-27T03:35:16.734186",
  "overall_pass": false,
  "audit_checks": [
    {
      "check_name": "completeness",
      "passed": true,
      "details": {
        "null_counts": {
          "name": 0,
          "email": 0,
          "country": 0,
          "amount": 0
        }
      }
    },
    {
      "check_name": "uniqueness",
      "passed": true,
      "details": {
        "duplicate_emails": []
      }
    },
    {
      "check_name": "referential_integrity",
      "passed": false,
      "details": {
        "orphan_customer_ids": [
          2,
          6
        ]
      }
    },
    {
      "check_name": "reconciliation",
      "passed": false,
      "details": {
        "raw_count": 5,
        "transformed_count": 1,
        "delta": 4
      }
    }
  ],
  "rejected_at_gate": {
    "customers": 3,
    "transactions": 1
  }
}


## 6. Mocked Alert Dispatch

If `overall_pass` is False, both alert channels are called.  The bodies are mocked
(no network or SMTP dependency), but the function signatures are production-ready.

In [7]:
from etl_challenge.alerts.dispatcher import send_email, send_webhook

if not report['overall_pass']:
    send_webhook(report)
    send_email(
        subject='ETL Audit Failed',
        body=json.dumps(report, indent=2),
    )
    print('Alerts dispatched (mocked — no network calls made).')
else:
    print('All checks passed — no alerts needed.')

Alerts dispatched (mocked — no network calls made).
